# FHIR Encounter Bronze-to-Silver Transformation

## Purpose

In this notebook, I transform raw FHIR Encounter resources from the Bronze
layer into a structured and analytics-ready Silver Delta table.

### Source

`health_insurance.bronze.fhir_encounter_raw`

### Target

`health_insurance.silver.fhir_encounter`

### What I am doing in this transformation

- I infer the combined FHIR Encounter schema using a Serverless-compatible approach.
- I parse the raw JSON stored in Bronze into structured Spark data.
- I extract encounter identifiers, status, class, and type.
- I safely handle optional or empty FHIR arrays such as `type` and `participant`.
- I parse Patient, Practitioner, and Organization references into usable identifiers.
- I convert encounter start and end values to Spark timestamps.
- I derive encounter duration in minutes when both timestamps are available.
- I standardize selected categorical values.
- I preserve Bronze ingestion metadata and add Silver transformation metadata.

not enforcing formal data-quality rules in this notebook. Those rules
will be attached to the production Lakeflow transformation so validation
happens while the Silver dataset is being produced.




In [0]:
# defining the source and target tables used by this transformation.

CATALOG = "health_insurance"

SOURCE_TABLE = f"{CATALOG}.bronze.fhir_encounter_raw"
TARGET_TABLE = f"{CATALOG}.silver.fhir_encounter"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)


In [0]:
# loading the Bronze FHIR Encounter table and inspecting its current shape.

encounter_bronze_df = spark.table(SOURCE_TABLE)

print(f"Rows: {encounter_bronze_df.count():,}")
print(f"Columns: {len(encounter_bronze_df.columns)}")

encounter_bronze_df.printSchema()

display(encounter_bronze_df.limit(5))


## Inferring the FHIR Encounter JSON schema

Because I am using Databricks Serverless, I avoid Spark RDD APIs.

I infer the schema directly from the Bronze `raw_json` column with
`schema_of_json_agg`. I use the aggregate form because FHIR fields are optional
and can vary between Encounter resources, so the resulting schema reflects the
combined structure currently present in Bronze.


In [0]:
# inferring a combined FHIR Encounter schema without using RDD operations.

schema_result = spark.sql(f'''
    SELECT schema_of_json_agg(raw_json) AS encounter_schema
    FROM {SOURCE_TABLE}
''').first()

encounter_schema = schema_result["encounter_schema"]

print("FHIR Encounter schema:")
print(encounter_schema)


In [0]:
# parsing each raw Encounter JSON string into a structured Spark column.

from pyspark.sql import functions as F

encounter_parsed_df = (
    encounter_bronze_df
    .withColumn(
        "encounter",
        F.from_json(
            F.col("raw_json"),
            encounter_schema
        )
    )
)

encounter_parsed_df.select("encounter.*").printSchema()


## Extracting the core Encounter attributes

I now select the Encounter fields required by the conformed Silver model while
temporarily retaining the nested structures and arrays that I still need to
flatten.

FHIR fields are optional, so I allow missing values to remain `NULL` rather
than rejecting records during structural transformation.


In [0]:
# extracting the core Encounter attributes and nested structures.

encounter_core_df = (
    encounter_parsed_df
    .select(
        F.col("encounter.id").alias("encounter_id"),
        F.col("encounter.status").alias("status"),

        F.col("encounter.class").alias("class_struct"),
        F.col("encounter.type").alias("type_array"),

        F.col("encounter.subject.reference").alias("patient_reference"),
        F.col("encounter.participant").alias("participant_array"),

        F.col("encounter.period.start").alias("start_datetime_raw"),
        F.col("encounter.period.end").alias("end_datetime_raw"),

        F.col("encounter.serviceProvider.reference").alias(
            "organization_reference"
        ),

        "_ingested_at",
        "_source_system",
        "_resource_type"
    )
)


## Extracting Encounter class

FHIR Encounter class is stored as a coding structure rather than an array in
the records currently present in Bronze.

I extract the class code and its coding system directly from that structure.


In [0]:
# extracting the Encounter class and its coding system.

encounter_class_df = (
    encounter_core_df

    .withColumn(
        "encounter_class",
        F.col("class_struct.code")
    )

    .withColumn(
        "encounter_class_system",
        F.col("class_struct.system")
    )
)


## Extracting the primary Encounter type safely

FHIR allows an Encounter to contain multiple `type` values, and each type can
contain multiple coding entries.

For my conformed Silver Encounter table, I keep the first available type and
the first available coding within it.

I use null-safe `get(array, 0)` access so an empty `type` or `coding` array
produces `NULL` instead of failing the transformation.


In [0]:
# extracting the primary Encounter type with null-safe array access.

encounter_type_df = (
    encounter_class_df

    .withColumn(
        "primary_type",
        F.expr("get(type_array, 0)")
    )

    .withColumn(
        "encounter_type_code",
        F.expr("get(primary_type.coding.code, 0)")
    )

    .withColumn(
        "encounter_type",
        F.expr("get(primary_type.coding.display, 0)")
    )

    .withColumn(
        "encounter_type_system",
        F.expr("get(primary_type.coding.system, 0)")
    )
)


## Extracting the primary Practitioner reference safely

FHIR `participant` is also an optional array and can contain more than one
participant.

For this Silver entity, I keep one row per Encounter and use the first
available participant as the primary practitioner reference.

If no participant exists, `get(participant_array, 0)` returns `NULL` instead of
causing an array-index failure.


In [0]:
# extracting the first available participant and practitioner reference safely.

encounter_participant_df = (
    encounter_type_df

    .withColumn(
        "primary_participant",
        F.expr("get(participant_array, 0)")
    )

    .withColumn(
        "practitioner_reference",
        F.col("primary_participant.individual.reference")
    )
)


## Parsing FHIR references

FHIR stores relationships as strings such as:

`Patient/<id>`

`Practitioner/<id>`

`Organization/<id>`

I extract only the identifier when the reference has the expected prefix.
Missing or unexpected references remain `NULL` instead of becoming empty
strings.


In [0]:
# converting FHIR references into clean identifiers.

encounter_refs_df = (
    encounter_participant_df

    .withColumn(
        "patient_id",
        F.when(
            F.col("patient_reference").startswith("Patient/"),
            F.regexp_extract(
                F.col("patient_reference"),
                r"^Patient/(.+)$",
                1
            )
        ).otherwise(F.lit(None).cast("string"))
    )

    .withColumn(
        "practitioner_id",
        F.when(
            F.col("practitioner_reference").startswith("Practitioner/"),
            F.regexp_extract(
                F.col("practitioner_reference"),
                r"^Practitioner/(.+)$",
                1
            )
        ).otherwise(F.lit(None).cast("string"))
    )

    .withColumn(
        "organization_id",
        F.when(
            F.col("organization_reference").startswith("Organization/"),
            F.regexp_extract(
                F.col("organization_reference"),
                r"^Organization/(.+)$",
                1
            )
        ).otherwise(F.lit(None).cast("string"))
    )
)


## Converting Encounter timestamps

I convert the raw FHIR encounter period values into Spark timestamps so they
can be used consistently for comparisons, duration calculations, filtering,
and downstream analytical models.


In [0]:
# converting the raw Encounter period values to Spark timestamps.

encounter_typed_df = (
    encounter_refs_df

    .withColumn(
        "start_datetime",
        F.to_timestamp("start_datetime_raw")
    )

    .withColumn(
        "end_datetime",
        F.to_timestamp("end_datetime_raw")
    )
)


## Deriving Encounter duration

When both start and end timestamps are available, I calculate the Encounter
duration in minutes.

If either timestamp is missing, I keep the duration as `NULL`. I do not try to
repair or reject the record here. Logical validation such as an end timestamp
occurring before the start timestamp will be handled by Lakeflow expectations
during productionization.


In [0]:
# deriving Encounter duration only when both timestamps are available.

encounter_enriched_df = (
    encounter_typed_df

    .withColumn(
        "encounter_duration_minutes",
        F.when(
            F.col("start_datetime").isNotNull()
            & F.col("end_datetime").isNotNull(),
            (
                F.col("end_datetime").cast("long")
                - F.col("start_datetime").cast("long")
            ) / 60
        ).otherwise(F.lit(None).cast("double"))
    )

    .withColumn(
        "encounter_duration_minutes",
        F.col("encounter_duration_minutes").cast("int")
    )
)


## Standardizing Encounter attributes

I standardize selected categorical values so the Silver table contains a
consistent representation for downstream joins and analytics.

I am only standardizing the representation here. I am not dropping or failing
records in this notebook.


In [0]:
# standardizing the Encounter status and class values.

encounter_standardized_df = (
    encounter_enriched_df

    .withColumn(
        "status",
        F.upper(F.trim("status"))
    )

    .withColumn(
        "encounter_class",
        F.upper(F.trim("encounter_class"))
    )
)


## Building the final Silver Encounter dataset

I now remove the temporary nested structures and helper fields and keep the
conformed attributes required by the Silver Encounter entity.

I preserve the Bronze ingestion metadata and add `_silver_transformed_at` so
the row can be traced through the Medallion architecture.


In [0]:
# building the final conformed Silver Encounter dataset.

encounter_silver_df = (
    encounter_standardized_df

    .select(
        "encounter_id",
        "patient_id",
        "practitioner_id",
        "organization_id",

        "status",

        "encounter_class",
        "encounter_class_system",

        "encounter_type_code",
        "encounter_type",
        "encounter_type_system",

        "start_datetime",
        "end_datetime",
        "encounter_duration_minutes",

        "_source_system",
        "_resource_type",
        "_ingested_at"
    )

    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp()
    )
)


In [0]:
# inspecting the final Silver Encounter schema and a small result sample.

encounter_silver_df.printSchema()

display(
    encounter_silver_df.limit(20)
)


## Profiling optional and relationship fields

Before persisting the table, I profile several relationship and encounter
attributes so I can understand the completeness of the incoming FHIR data.

These are descriptive checks only. I will convert the appropriate technical and
business rules into Lakeflow expectations during the production pipeline stage.


In [0]:
# profiling missing Encounter identifiers, references, and timing fields.

encounter_silver_df.select(
    F.sum(F.col("encounter_id").isNull().cast("int")).alias("missing_encounter_id"),
    F.sum(F.col("patient_id").isNull().cast("int")).alias("missing_patient_id"),
    F.sum(F.col("practitioner_id").isNull().cast("int")).alias("missing_practitioner_id"),
    F.sum(F.col("organization_id").isNull().cast("int")).alias("missing_organization_id"),
    F.sum(F.col("encounter_type_code").isNull().cast("int")).alias("missing_encounter_type_code"),
    F.sum(F.col("start_datetime").isNull().cast("int")).alias("missing_start_datetime"),
    F.sum(F.col("end_datetime").isNull().cast("int")).alias("missing_end_datetime")
).show()


In [0]:
# reconciling Bronze and Silver row counts before persistence.

bronze_count = encounter_bronze_df.count()
silver_count = encounter_silver_df.count()

print(f"Bronze Encounters: {bronze_count:,}")
print(f"Silver Encounters: {silver_count:,}")
print(f"Difference: {bronze_count - silver_count:,}")


## Persisting the Silver table

At this development stage, I overwrite the Silver table so the notebook remains
repeatable while I validate the transformation logic.

When I productionize this transformation in Lakeflow, the execution pattern
will become incremental and quality expectations will be evaluated while
Silver is being produced. The parsing and conformance logic developed here is
intended to remain reusable.


In [0]:
# persisting the conformed Encounter dataset as a Delta table in Unity Catalog.

(
    encounter_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TARGET_TABLE)
)

print(f"Created Silver table: {TARGET_TABLE}")


In [0]:
%sql
-- verifying the number of Encounter records written to Silver.

SELECT COUNT(*) AS encounter_count
FROM health_insurance.silver.fhir_encounter;


In [0]:
%sql
-- reviewing Encounter type and duration as a sanity check.

SELECT
    encounter_class,
    encounter_type,
    COUNT(*) AS encounter_count,
    ROUND(AVG(encounter_duration_minutes), 2) AS avg_duration_minutes
FROM health_insurance.silver.fhir_encounter
GROUP BY encounter_class, encounter_type
ORDER BY encounter_count DESC;


## Transformation Result

successfully transformed the raw FHIR Encounter resources from Bronze into a
structured Silver Delta table.

### Source

`health_insurance.bronze.fhir_encounter_raw`

### Target

`health_insurance.silver.fhir_encounter`

### Transformations I applied

- I inferred the combined FHIR Encounter schema using a Serverless-compatible
  DataFrame/SQL approach.
- I parsed `raw_json` into structured Spark data.
- I extracted Encounter identifiers, status, class, and type.
- I used null-safe array access for optional `type`, coding, and participant arrays.
- I parsed Patient, Practitioner, and Organization references into clean identifiers.
- I converted Encounter period values into Spark timestamps.
- I derived Encounter duration in minutes only when both timestamps were available.
- I standardized selected categorical values.
- I preserved Bronze lineage metadata and added a Silver transformation timestamp.
- I profiled missing relationship, type, and timing fields without filtering records.
- I reconciled Bronze and Silver row counts before persistence.

### Data-quality boundary

not rejecting records in this notebook.

Formal rules such as required Encounter IDs, required Patient references,
supported statuses, logical start/end timestamps, and relationship integrity
will be attached as Lakeflow expectations when I productionize the
Bronze-to-Silver pipeline.

### Architecture

SMART FHIR API  
↓  
Auto Loader-managed Bronze ingestion  
↓  
`health_insurance.bronze.fhir_encounter_raw`  
↓  
FHIR parsing and conformance  
↓  
`health_insurance.silver.fhir_encounter`  
↓  
Lakeflow expectations / validated Silver  
↓  
Gold analytical model



### Status

**FHIR Encounter Bronze-to-Silver transformation: COMPLETE**
